In [0]:
spark.sql("USE CATALOG relp")
spark.sql("USE SCHEMA bronze")
spark.sql("USE SCHEMA silver")

###Imports

In [0]:
import importlib
import pipeline.silver.transformation as transformation

importlib.reload(transformation)

from pipeline.silver.transformation import *
print("=" * 60)
print("🚀 SILVER LAYER STARTED")
print("=" * 60)

### 1️⃣ Read Bronze Tables


In [0]:
df_trans    = spark.table("bronze.bronze_transactions")
df_cust     = spark.table("bronze.bronze_customers")
df_prop     = spark.table("bronze.bronze_properties")
df_agent    = spark.table("bronze.bronze_agents")
df_listings = spark.table("bronze.bronze_listings")

print("\n📊 Bronze Counts:")
print(f"Transactions : {df_trans.count()}")
print(f"Customers    : {df_cust.count()}")
print(f"Properties   : {df_prop.count()}")
print(f"Agents       : {df_agent.count()}")
print(f"Listings     : {df_listings.count()}")

###2️⃣ Apply Cleaning

In [0]:
print("\n🔧 Cleaning Tables...")

df_trans_clean    = clean_transactions(df_trans)
df_cust_clean     = clean_customers(df_cust)
df_prop_clean     = clean_properties(df_prop)
df_agent_clean    = clean_agents(df_agent)
df_listings_clean = clean_listings(df_listings)

#### 🆕 Save Clean Dimension Tables

In [0]:
print("\n💾 Saving Clean Dimension Tables...")

df_cust_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.silver_customers")

df_prop_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.silver_properties")

df_agent_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.silver_agents")

df_listings_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.silver_listings")

### 3️⃣ Build Final Silver Fact Table

In [0]:
print("\n🔗 Building Integrated Silver Fact Table...")

df_silver = build_silver_fact(
    df_trans_clean,
    df_cust_clean,
    df_prop_clean,
    df_agent_clean,
    df_listings_clean
)


### 4️⃣ Preview

In [0]:
print("\n📊 Silver Preview:")

display(df_silver.limit(10))

### 5️⃣ Write Silver Table

In [0]:
print("\n💾 Writing Silver Table...")

df_silver.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.silver_real_estate_fact")

## 🧪 FINAL SILVER VALIDATION REPORT

In [0]:
from pyspark.sql.functions import count, when

print("=" * 70)
print("🧪 FINAL SILVER VALIDATION REPORT")
print("=" * 70)

### 1️⃣ Load Table

In [0]:
df_fact = spark.table("silver.silver_real_estate_fact")

### 2️⃣ Schema

In [0]:
print("\n📌 FACT TABLE SCHEMA:")
df_fact.printSchema()

### 3️⃣ Sample Data

In [0]:
print("\n📊 SAMPLE DATA:")
display(df_fact.limit(10))


### 4️⃣ Core Count



In [0]:
silver_count = df_fact.count()

print(f"\n📊 Total Records: {silver_count}")
print(f"📊 Total Columns: {len(df_silver.columns)}")


### Date Validation

In [0]:
print("\n📅 Date Validation:")

display(
    df_fact.select(
        (col("deal_date") > current_date()).alias("future_date")
    ).groupBy("future_date").count()
)

### City Consistency

In [0]:
print("\n🌍 City Consistency Check:")

display(
    df_fact.groupBy("city_variation_flag").count()
)

### 5️⃣ Null Check

In [0]:
print("\n⚠️ Null Check:")

nulls = df_fact.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
])

display(nulls)


### 6️⃣ Duplicate Check

In [0]:
print("\n🧾 Duplicate Check:")

dups = df_fact.groupBy("transaction_id").count().filter("count > 1")

if dups.count() > 0:
    print("❌ Duplicates Found")
    display(dups)
else:
    print("✅ No duplicates found")


### 7️⃣ Business Rule Check

In [0]:
print("\n💰 Business Rule Validation:")

display(
    df_fact.select(
        (col("commission_amount") > col("deal_price")).alias("invalid_commission")
    ).groupBy("invalid_commission").count()
)


###8️⃣ Join Integrity Check

In [0]:
print("\n🔍 Join Validation:")

missing_cust  = df_fact.filter(col("home_city").isNull()).count()
missing_prop  = df_fact.filter(col("property_type").isNull()).count()
missing_agent = df_fact.filter(col("agent_city").isNull()).count()

join_success = 100 - (
    (missing_cust + missing_prop + missing_agent) / (3 * silver_count) * 100
)

print(f"Missing Customers : {missing_cust} ({round(missing_cust/silver_count*100,2)}%)")
print(f"Missing Properties: {missing_prop} ({round(missing_prop/silver_count*100,2)}%)")
print(f"Missing Agents    : {missing_agent} ({round(missing_agent/silver_count*100,2)}%)")

print(f"\n✅ Join Success Rate: {round(join_success,2)}%")

###9️⃣ Catalog Check

In [0]:
print("\n📂 Tables in Silver Schema:")

display(
    spark.sql("SHOW TABLES IN silver")
)

## FINAL STATUS

In [0]:
print("\n" + "=" * 70)
print("✅ SILVER LAYER COMPLETE & VALIDATED")
print("=" * 70)

In [0]:
display(df_fact.groupBy("income_band").count())

##📊 FINAL SILVER SUMMARY

In [0]:
from pyspark.sql.functions import col, current_date

print("\n" + "="*60)
print("📊 FINAL SILVER SUMMARY")
print("="*60)

# =========================
# Dynamic Counts
# =========================
fact_count = df_fact.count()
bronze_df = spark.table("bronze.bronze_transactions")
bronze_count = bronze_df.count()

print(f"Total Records        : {fact_count}")
print(f"Columns              : {len(df_fact.columns)}")

# =========================
# Retention / Drop Metrics
# =========================
retention_rate = (fact_count / bronze_count) * 100
rows_dropped = bronze_count - fact_count
drop_pct = (rows_dropped / bronze_count) * 100

print(f"Data Retention Rate  : {retention_rate:.2f}%")
print(f"Rows Dropped         : {rows_dropped}")
print(f"Drop Percentage      : {drop_pct:.2f}%")

# =========================
# Root Cause Analysis
# =========================
dropped_df = bronze_df.join(df_fact, "transaction_id", "left_anti")

dropped_count = dropped_df.count()

print("\n🔍 Dropped Records Analysis:")
print(f"Total Dropped Rows   : {dropped_count}")

# 🔥 Drop Reason Analysis
invalid_price_rows = dropped_df.filter(col("deal_price").isNull()).count()

print("\n📉 Drop Reason Analysis:")
print(f"Rows with NULL deal_price: {invalid_price_rows}")

display(dropped_df.limit(5))

# =========================
# Validation Checks
# =========================
dup_count = df_fact.groupBy("transaction_id").count().filter("count > 1").count()

null_critical = df_fact.filter(
    col("transaction_id").isNull() |
    col("customer_id").isNull() |
    col("property_id").isNull() |
    col("agent_id").isNull()
).count()

future_dates = df_fact.filter(col("deal_date") > current_date()).count()
null_dates = df_fact.filter(col("deal_date").isNull()).count()

invalid_commission = df_fact.filter(
    col("commission_amount") > col("deal_price")
).count()

# 🔥 Listing validation (NEW – makes you stand out)
missing_listing = df_fact.filter(col("listing_channel").isNull()).count()

print("\nKey Validations:")

print(f"{'✔' if dup_count == 0 else '❌'} No duplicates in transaction_id ({dup_count})")
print(f"{'✔' if null_critical == 0 else '❌'} No nulls in critical keys ({null_critical})")
print(f"{'✔' if invalid_commission == 0 else '❌'} Commission rules validated ({invalid_commission})")
print(f"✔ Data retention {retention_rate:.2f}%")
print(f"{'✔' if future_dates == 0 else '❌'} No future dates ({future_dates})")
print(f"{'✔' if null_dates == 0 else '❌'} No null deal dates ({null_dates})")

print(f"\n🔗 Listing Join Check:")
print(f"Missing Listings     : {missing_listing} ({round(missing_listing/fact_count*100,2)}%)")

# =========================
# Schema
# =========================
print("\n📌 FACT TABLE SCHEMA:")
df_fact.printSchema()

# =========================
# Column Categories
# =========================
print("\n📌 Column Categories:")
print("✔ Keys: transaction_id, customer_id, property_id, agent_id")
print("✔ Measures: deal_price, commission_amount")
print("✔ Dimensions: city, property_type, segment, etc.")
print("✔ Derived: year_month, price_category")

# =========================
# Final Status
# =========================
if (dup_count == 0 and 
    null_critical == 0 and 
    invalid_commission == 0 and 
    future_dates == 0):
    
    print("\n✅ SILVER LAYER: PRODUCTION READY")
else:
    print("\n⚠️ SILVER LAYER: NEEDS ATTENTION")

print("="*60)